# exp-02 Batched actions

**Question.** When a turn needs several edits, how do LLM steps, tokens, latency and pass rate change if the
model sends all edits in one response (arm B), or as one list in a single `apply_edits` call (arm C), instead of
one edit per step (arm A, today's app)?

**Dataset.** `data/examples.json`, 300 single-turn examples, gold `ops` (1–4 edits). The run cell calls the LLM
only when `RUN_PAID = True`.

In [ ]:
# Setup: works whether the notebook's folder or the repo root is the working directory.
import json, sys
from pathlib import Path
HERE = Path.cwd()
if HERE.name != "exp02_batched_actions":
    HERE = next(p for p in [Path.cwd() / "eng-experiments" / "exp02_batched_actions", Path.cwd() / "exp02_batched_actions"] if p.exists())
sys.path[:0] = [str(HERE), str(HERE.parent)]
import os; os.environ.setdefault("TRACE_PRINT", "0")   # no per-span printing during runs

import pandas as pd
from IPython.display import Markdown, display
import common
import run, variants
pd.set_option("display.max_colwidth", 120)

## Settings

In [ ]:
RUN_PAID = False          # True = really call the LLM (costs money)
FIRST = None              # e.g. 3 for a smoke run; None = all 300 examples
MODEL = common.DEFAULT_MODEL
examples = common.load_examples(first=FIRST)
print(len(examples), "examples; model", MODEL)
pd.Series([len(e["ops"]) for e in examples]).value_counts().sort_index().rename("examples by number of gold edits")

## The arms

In [ ]:
arms = run.arms()
pd.DataFrame([{"arm": v.name, "prompt": Path(v.prompt_file).stem, "setup": v.notes} for v in arms]).set_index("arm")

In [ ]:
# Arm C's tool as the LLM sees it
print(json.dumps(variants.APPLY_EDITS["parameters"], indent=1)[:1500])

In [ ]:
turns = len(examples) * len(arms)
print(f"{turns} agent turns, about ${turns * run.USD_PER_TURN:.2f}")

## Run (paid)

In [ ]:
if RUN_PAID:
    rows = common.run_examples("exp02", run.arms(), examples, MODEL, HERE / "results")
else:
    print("RUN_PAID is False: nothing was called.")

## Results (latest run in `results/`)

Also rewrites the tables in `README.md`.

In [ ]:
path = common.latest(HERE / "results", "exp02-*.json")
if path:
    data = json.loads(path.read_text())
    rows, arm_names = data["rows"], [a["name"] for a in data["meta"]["arms"]]
    display(common.frame(rows, arm_names))
    display(Markdown(run.report(path)))
else:
    print("No results yet.")

In [ ]:
# Per example: the edits each arm made vs the gold ops (examples with 2+ gold edits)
if path:
    ex = {e["id"]: e for e in common.load_examples()}
    df = pd.DataFrame([{"example": r["example"], "arm": r["arm"], "pass": r["pass"], "steps": r["steps"],
                        "edits": "; ".join(run.edit_ops(r)), "gold": "; ".join(ex[r["example"]]["ops"])}
                       for r in rows if len(ex[r["example"]]["ops"]) >= 2])
    display(df.pivot(index=["example", "gold"], columns="arm", values=["steps", "edits"]))